In [ ]:
import logging
import time
from pathlib import Path
from collections import deque
from typing import Dict, Any
import numpy as np

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import src.gymnasium_envs.convex_optimization_env

PROJECT_ROOT = Path().resolve().parent.parent

log_dir = PROJECT_ROOT / "logs"
model_dir = PROJECT_ROOT / "models"

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

Adding status logging

In [ ]:
class StatusCallback(BaseCallback):
    def __init__(self, window_size=1000):
        super().__init__()
        self.window = deque(maxlen=window_size)
        self.steps_to_converge = deque(maxlen=window_size)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            status = info.get("status")
            if status == "converged":
                self.window.append(1)
                self.steps_to_converge.append(info["iteration"])
            elif status == "diverged":
                self.window.append(-1)

        if len(self.window) > 0:
            self.logger.record("custom/converge_rate",
                self.window.count(1) / len(self.window))
            self.logger.record("custom/diverge_rate",
                self.window.count(-1) / len(self.window))

        if len(self.steps_to_converge) > 0:
            self.logger.record("custom/mean_steps_to_converge",
                np.mean(self.steps_to_converge))  # чем меньше тем лучше

        return True

class BestConvergeCallback(BaseCallback):
    def __init__(self, vec_normalize_env, model_save_path, vec_normalize_save_path, verbose=1):
        super().__init__(verbose)
        self.vec_normalize_env = vec_normalize_env
        self.model_save_path = model_save_path
        self.vec_normalize_save_path = vec_normalize_save_path
        self.best_converge_rate = -np.inf

    def _on_step(self) -> bool:
        converge_rate = self.logger.name_to_value.get("custom/converge_rate", None)

        if converge_rate is not None and converge_rate > self.best_converge_rate:
            self.best_converge_rate = converge_rate
            self.model.save(self.model_save_path)
            self.vec_normalize_env.save(self.vec_normalize_save_path)
            if self.verbose:
                logger.info(f"New best converge_rate: {converge_rate:.4f}")
        return True

Let's create a function for training

In [6]:
def train_model(config: Dict[str, Any], log_dir: str = "logs", model_dir: str = "models", add_time_penalty : bool = False):
    in_features_name = "any" if config["in_features"] is None else config["in_features"]
    logger.info(f"Starting training for {in_features_name}D task. Timesteps: {config['timesteps']}, Envs: {config['n_envs']}")

    start_time    = time.time()
    model_path    = Path(f"{model_dir}/{in_features_name}d_convex")
    best_model_path = Path(f"{model_dir}/{in_features_name}d_convex_best")
    stats_path    = Path(f"{model_dir}/{in_features_name}d_convex_vec_normalize.pkl")

    model_path.parent.mkdir(parents=True, exist_ok=True)
    best_model_path.mkdir(parents=True, exist_ok=True)

    vec_env = None
    try:
        vec_env = make_vec_env(
            "convex_optimization_env/ConvexOptimization-v1",
            n_envs=config["n_envs"],
            env_kwargs={"in_features": config["in_features"],
                        "add_time_penalty" : add_time_penalty}
        )
        vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True)

        best_converge_callback = BestConvergeCallback(
            vec_normalize_env=vec_env,
            model_save_path=str(best_model_path / "best_model"),
            vec_normalize_save_path=str(best_model_path / "best_vec_normalize.pkl"),
            verbose=1
        )

        model = PPO(
            "MultiInputPolicy",
            vec_env,
            verbose=1,
            tensorboard_log=f"{log_dir}/{in_features_name}d/",
            n_steps=config["n_steps"],
            n_epochs=config["n_epochs"],
            batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            gamma=config["gamma"],
            clip_range=config["clip_range"],
            ent_coef=config["ent_coef"],
            policy_kwargs=config["policy_kwargs"]
        )

        model.learn(
            total_timesteps=config["timesteps"],
            tb_log_name=f"convex_{in_features_name}d",
            callback=CallbackList([
                StatusCallback(),
                best_converge_callback
            ])
        )

        # Сохраняем финальную модель отдельно
        model.save(str(model_path))
        vec_env.save(str(stats_path))

        elapsed_time = (time.time() - start_time) / 60
        logger.info(f"Training for {in_features_name}D completed. Duration: {elapsed_time:.2f} min.")
        logger.info(f"Final model saved to: {model_path}")
        logger.info(f"Best model saved to: {best_model_path / 'best_model'}")

    except Exception as e:
        logger.error(f"Error during training for {in_features_name}D: {str(e)}", exc_info=True)
    finally:
        if vec_env is not None:
            vec_env.close()

In [ ]:
config_base = {
    "in_features"   : None,
    "timesteps"     : 5_000_000, 
    "n_envs"        : 32,
    "n_steps"       : 2048,
    "n_epochs"      : 10, 
    "batch_size"    : 256, 
    "learning_rate" : 3e-4,
    "gamma"         : 0.99,
    "clip_range"    : 0.2,
    "ent_coef"      : 0.01,
    "policy_kwargs" : {
        "net_arch": dict(pi=[64, 64], vf=[64, 64])
    }
}

Let's train a 2D model

In [ ]:
config = config_base
config["in_features"] = 2

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = 5

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = 10

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = 100

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir)

In [7]:
config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir, add_time_penalty = True)

2026-05-28 16:37:25 [INFO] Starting training for anyD task. Timesteps: 5000000, Envs: 32
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/anyd/convex_anyd_2


2026-05-28 16:37:29 [INFO] New best converge_rate: 0.0000
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\numpy\linalg\_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:87: RuntimeWarning: overflow encountered in matmul
  return float(x.T @ self._A @ x + self._b @ x + self._c)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\gymnasium_envs\convex_optimization_env\envs\convex_optimization_v1.py:323: RuntimeWarning: overflow encountered in dot
  cos_sim = np.dot(grad_unit, prev_grad_unit)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:92: RuntimeWarning: overflow encountered in matmul
  return 2 * self._A @ x + self._b
C:\Users\Lolik\Documents\GitHub\Reinfor

---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 642      |
|    ep_rew_mean     | -263     |
| time/              |          |
|    fps             | 4820     |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 592          |
|    ep_rew_mean          | -243         |
| time/                   |              |
|    fps                  | 2652         |
|    iterations           | 2            |
|    time_elapsed         | 49           |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-05-28 16:47:32 [INFO] New best converge_rate: 0.0025


...

2026-05-28 17:21:50 [INFO] New best converge_rate: 0.2999
2026-05-28 17:21:54 [INFO] New best converge_rate: 0.3009
2026-05-28 17:21:56 [INFO] New best converge_rate: 0.3019


-------------------------------------------
| custom/                   |             |
|    converge_rate          | 0.302       |
|    diverge_rate           | 0.698       |
|    mean_steps_to_converge | 425         |
| rollout/                  |             |
|    ep_len_mean            | 959         |
|    ep_rew_mean            | -40.2       |
| time/                     |             |
|    fps                    | 1864        |
|    iterations             | 76          |
|    time_elapsed           | 2670        |
|    total_timesteps        | 4980736     |
| train/                    |             |
|    approx_kl              | 0.002905435 |
|    clip_fraction          | 0.0207      |
|    clip_range             | 0.2         |
|    entropy_loss           | 0.0427      |
|    explained_variance     | 0.91        |
|    learning_rate          | 0.0003      |
|    loss                   | 0.00635     |
|    n_updates              | 750         |
|    policy_gradient_loss   | -0

2026-05-28 17:22:23 [INFO] New best converge_rate: 0.3029
2026-05-28 17:22:26 [INFO] New best converge_rate: 0.3039
2026-05-28 17:22:27 [INFO] New best converge_rate: 0.3048
2026-05-28 17:22:27 [INFO] New best converge_rate: 0.3058
2026-05-28 17:22:29 [INFO] New best converge_rate: 0.3068
2026-05-28 17:22:29 [INFO] New best converge_rate: 0.3078
2026-05-28 17:22:31 [INFO] New best converge_rate: 0.3088
2026-05-28 17:22:35 [INFO] New best converge_rate: 0.3098
2026-05-28 17:22:37 [INFO] New best converge_rate: 0.3107
2026-05-28 17:22:37 [INFO] New best converge_rate: 0.3117


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.312        |
|    diverge_rate           | 0.688        |
|    mean_steps_to_converge | 426          |
| rollout/                  |              |
|    ep_len_mean            | 933          |
|    ep_rew_mean            | -37.2        |
| time/                     |              |
|    fps                    | 1861         |
|    iterations             | 77           |
|    time_elapsed           | 2710         |
|    total_timesteps        | 5046272      |
| train/                    |              |
|    approx_kl              | 0.0023025917 |
|    clip_fraction          | 0.02         |
|    clip_range             | 0.2          |
|    entropy_loss           | 0.0513       |
|    explained_variance     | 0.928        |
|    learning_rate          | 0.0003       |
|    loss                   | -0.00638     |
|    n_updates              | 760          |
|    polic

2026-05-28 17:22:59 [INFO] Training for anyD completed. Duration: 45.57 min.
2026-05-28 17:22:59 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\anyd_convex
2026-05-28 17:22:59 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\anyd_convex_best\best_model
